# Tarea 2: disponibilidad y evolución temporal de puntos de acceso wifi

Este notebook acompaña los cálculos de la Tarea 2. El desarrollo matemático y las interpretaciones deben quedar en el informe.

## Integrantes

- TODO: nombre completo y rol USM del integrante 1.
- TODO: nombre completo y rol USM del integrante 2, si corresponde.

## Uso

Complete los TODO, ejecute las celdas en orden y guarde el notebook con los resultados visibles.


## Bibliotecas necesarias

Si falta alguna biblioteca, instálela con:

```python
%pip install numpy matplotlib
```

## Sección 1.1 del enunciado: Diagrama y matriz de transición

Use el orden de estados $(F,D,O)$ indicado en el enunciado.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb

np.set_printoptions(precision=5, suppress=True)

orden_estados = ["F", "D", "O"]

# TODO: Complete/derive las otras transiciones usando las razones del enunciado.
# Interpretación de ejemplo: p_fd es la probabilidad de iniciar en estado F y terminar en estado D
p_ff = None
p_dd = None
p_oo = None
p_fd = None
p_fo = None
p_df = None
p_do = None
p_of = None
p_od = None

P = np.array([
    [p_ff, p_fd, p_fo],
    [p_df, p_dd, p_do],
    [p_of, p_od, p_oo],
], dtype=float)

print("Orden de estados:", orden_estados)
print("Matriz P:\n", P)
print("Probabilidades entre cero y uno:", np.all((P >= 0) & (P <= 1)))
print("Suma de cada fila:", P.sum(axis=1))
print("Las filas suman uno:", np.allclose(P.sum(axis=1), 1))
print("F a O:", P[0, 2])
print("O a F:", P[2, 0])

## Sección 1.2 del enunciado: Distribución transiente y disponibilidad

Use el orden $(F,D,O)$ y el criterio de disponibilidad del enunciado.

In [ ]:
alpha_0 = np.array([0.0, 0.0, 1.0])
alpha_1 = alpha_0 @ P # @ representa multiplicación matricial
alpha_2 = alpha_1 @ P
dias = np.arange(1, 31)
alpha_evolucion = np.array([
    alpha_0 @ np.linalg.matrix_power(P, int(dia)) # matrix_power -> P^n
    for dia in dias
])
alpha_7 = alpha_evolucion[6]
alpha_30 = alpha_evolucion[29]

# TODO: indique con cero o uno si cada estado entrega conectividad.
# Para ello defínalo en la variable indicador_conectividad como un vector (lista python) 
indicador_conectividad = None
indicador_conectividad = np.asarray(indicador_conectividad, dtype=float)

disponibilidad_ap = alpha_evolucion @ indicador_conectividad

print("alpha_1:", alpha_1)
print("alpha_2:", alpha_2)
print("alpha_7:", alpha_7)
print("alpha_30:", alpha_30)
for dia in [1, 2, 7, 30]:
    print(f"Disponibilidad en el día {dia}: {disponibilidad_ap[dia - 1]:.6f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(dias, disponibilidad_ap, label="Disponibilidad del AP")
plt.scatter([1, 2, 7, 30], [disponibilidad_ap[0], disponibilidad_ap[1], disponibilidad_ap[6], disponibilidad_ap[29]])
plt.xlabel("Día")
plt.ylabel("Probabilidad de entregar conectividad")
plt.title("Evolución de la disponibilidad de un AP")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Sección 2.1 del enunciado: Primera visita y primer retorno

Parta desde el estado $O$.

In [ ]:
P_dos_pasos = np.linalg.matrix_power(P, 2)

# TODO: sume las probabilidades de las rutas de primera visita a F.
# Indique en el informe las rutas utilizadas y en el jupyter notebook a cuál
# ruta corresponde cada variable 
ruta_of1 = None 
ruta_of2 = None
primera_visita_f_dos = None

# TODO: sume las probabilidades de las rutas de primer retorno a O.
# Indique en el informe las rutas utilizadas y en el jupyter notebook a cuál
# ruta corresponde cada variable
ruta_oo1 = None
ruta_oo2 = None
primer_retorno_o_dos = None

print("Primera visita a F en el paso 2:", primera_visita_f_dos)
print("Entrada correspondiente de P al cuadrado:", P_dos_pasos[2, 0])
print("Primer retorno a O en el paso 2:", primer_retorno_o_dos)
print("Entrada de O a O en P al cuadrado:", P_dos_pasos[2, 2])

In [ ]:
pasos = np.arange(1, 31)
prob_en_f = np.array([np.linalg.matrix_power(P, int(n))[2, 0] for n in pasos])
primera_visita_f = np.zeros(len(pasos))

for n in pasos:
    suma_rutas_anteriores = 0.0
    # NOTE: El siguiente código debe implementar lo visto en la slide "Relación recursiva para primera visita", presentación 04
    for k in range(1, n):
        prob_permanece_f = np.linalg.matrix_power(P, int(n - k))[0, 0]
        # TODO: sume las rutas que llegaron por primera vez en k y luego terminan en F
    primera_visita_f[n - 1] = prob_en_f[n - 1] - suma_rutas_anteriores

print("Probabilidad de estar en F después de 30 días:", prob_en_f[-1])
print("Probabilidad de primera visita a F en el día 30:", primera_visita_f[-1])

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(pasos, prob_en_f, label="En F en el día n")
plt.plot(pasos, primera_visita_f, label="Primera visita a F en el día n")
plt.xlabel("Día n")
plt.ylabel("Probabilidad")
plt.title("Primera visita y probabilidad en n pasos")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Sección 2.2 del enunciado: Distribución estacionaria

Use el criterio de servicio de la Pregunta 1.

In [ ]:
sistema_pi = P.T - np.eye(3)
sistema_pi[-1, :] = 1.0
termino_derecho = np.array([0.0, 0.0, 1.0])
pi_ap = np.linalg.solve(sistema_pi, termino_derecho)
disponibilidad_estacionaria_ap = pi_ap @ indicador_conectividad

print("Distribución estacionaria del AP:", pi_ap)
print("Verificación pi por P:", pi_ap @ P)
print("Disponibilidad estacionaria del AP:", disponibilidad_estacionaria_ap)
print("Disponibilidad en el día 30:", disponibilidad_ap[29])

## Sección 3 del enunciado: Extensión a cinco AP

Complete los valores de $p$ y $r$.

In [ ]:
# TODO: complete p y r según lo indicado en el enunciado.
p = None
r = None
print("p:", p)
print("r:", r)

### Sección 3.2 del enunciado: Disponibilidad de la red

Calcule las disponibilidades transiente y estacionaria de la red.

In [ ]:
numero_ap = 5
P_cinco = np.zeros((numero_ap + 1, numero_ap + 1))

for disponibles_hoy in range(numero_ap + 1):
    for sobreviven in range(disponibles_hoy + 1):
        prob_sobreviven = (
            comb(disponibles_hoy, sobreviven)
            * (1 - p) ** sobreviven
            * p ** (disponibles_hoy - sobreviven)
        )
        for se_recuperan in range(numero_ap - disponibles_hoy + 1):
            prob_recuperacion = (
                comb(numero_ap - disponibles_hoy, se_recuperan)
                * r ** se_recuperan
                * (1 - r) ** (numero_ap - disponibles_hoy - se_recuperan)
            )
            disponibles_manana = sobreviven + se_recuperan
            P_cinco[disponibles_hoy, disponibles_manana] += prob_sobreviven * prob_recuperacion

print("Suma de cada fila:", P_cinco.sum(axis=1))
print("Las filas suman uno:", np.allclose(P_cinco.sum(axis=1), 1))
print("Matriz de transición para cinco AP:", P_cinco)

In [ ]:
alpha_cinco_0 = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 1.0])
alpha_cinco_30 = alpha_cinco_0 @ np.linalg.matrix_power(P_cinco, 30)

sistema_pi_cinco = P_cinco.T - np.eye(numero_ap + 1)
sistema_pi_cinco[-1, :] = 1.0
termino_cinco = np.zeros(numero_ap + 1)
termino_cinco[-1] = 1.0
pi_cinco = np.linalg.solve(sistema_pi_cinco, termino_cinco)

disponibilidad_red_30 = alpha_cinco_30[3:].sum()
disponibilidad_red_estacionaria = pi_cinco[3:].sum()

print("Distribución en el día 30:", alpha_cinco_30)
print("Distribución estacionaria:", pi_cinco)
print("Disponibilidad de la red en el día 30:", disponibilidad_red_30)
print("Disponibilidad estacionaria de la red:", disponibilidad_red_estacionaria)

In [ ]:
distribuciones_cinco = np.array([
    alpha_cinco_0 @ np.linalg.matrix_power(P_cinco, int(dia))
    for dia in dias
])
disponibilidad_red = distribuciones_cinco[:, 3:].sum(axis=1)

plt.figure(figsize=(7, 4))
plt.plot(dias, disponibilidad_red, label="Disponibilidad de la red")
plt.axhline(disponibilidad_red_estacionaria, color="tab:red", linestyle="--", label="Disponibilidad estacionaria")
plt.xlabel("Día")
plt.ylabel("Probabilidad de entregar servicio")
plt.title("Evolución de la disponibilidad de cinco AP")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()